# [STARTER] Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [1]:
%pip install -r ../requirements.txt
# Only needed for Udacity workspace
import importlib.util
import sys
import os
# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
# TODO: Load environment variables
load_dotenv()

True

### VectorDB Instance

In [3]:
# TODO: Instantiate your ChromaDB Client
# Choose any path you want
chroma_client = chromadb.PersistentClient(path="chromadb")

### Collection

In [4]:
# TODO: Pick one embedding function
# If picking something different than openai,
# make sure you use the same when loading it
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.getenv("OPENAI_API_KEY"),
    api_base=os.getenv("OPENAI_BASE_URL")
)

In [5]:
# TODO: Create a collection
# Choose any name you want
collection = chroma_client.create_collection(
    name="udaplay_project",
    embedding_function=embedding_fn,
    get_or_create=True
)

### Add documents

In [6]:
# Make sure you have a directory "project/starter/games"
data_dir = "games"

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)
    # You can change what text you want to index
    content = f"[{game['Platform']} - {game['Genre']}] {game['Name']} ({game['YearOfRelease']}) - {game['Description']}"
    # Use file name (like 001) as ID
    doc_id = os.path.splitext(file_name)[0]

    collection.add(
        ids=[doc_id],
        documents=[content],
        metadatas=[game]
    )

### Semantic search demonstration

In [7]:
results = collection.query(
    query_texts=["an open-world action adventure with theft and vehicles"],
    n_results=2,
    include=['metadatas', 'documents', 'distances']
)
results['documents'][0]

["[PlayStation 2 - Action-adventure] Grand Theft Auto: San Andreas (2004) - An expansive open-world game set in the fictional state of San Andreas, following the story of Carl 'CJ' Johnson.",
 "[PlayStation 4 - Action-adventure] Marvel's Spider-Man (2018) - An open-world superhero game that lets players swing through New York City as Spider-Man, battling iconic villains."]